# Q:2

In [1]:
corpus = [
    "the boy hugs the cat",
    "the boys are hugging the dogs",
    "the dogs are chasing the cats",
    "the dog and the cat sit quietly",
    "the boy is sitting on the dog"
]


# preprocess 

In [2]:
import re
from collections import Counter, defaultdict

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text.split()

# Create list of words
words = []
for sentence in corpus:
    words.extend(preprocess(sentence))

# Represent words as list of characters
tokenized = [[' '.join(list(w)) + ' </w>' for w in words]]
tokenized = [item.split() for sublist in tokenized for item in sublist]
tokenized[:10]


[['t', 'h', 'e', '</w>'],
 ['b', 'o', 'y', '</w>'],
 ['h', 'u', 'g', 's', '</w>'],
 ['t', 'h', 'e', '</w>'],
 ['c', 'a', 't', '</w>'],
 ['t', 'h', 'e', '</w>'],
 ['b', 'o', 'y', 's', '</w>'],
 ['a', 'r', 'e', '</w>'],
 ['h', 'u', 'g', 'g', 'i', 'n', 'g', '</w>'],
 ['t', 'h', 'e', '</w>']]

# compute pair counts

In [3]:
def get_pair_counts(words):
    pair_counts = Counter()
    symbol_counts = Counter()
    
    for word in words:
        symbol_counts.update(word)
        for i in range(len(word) - 1):
            pair_counts[(word[i], word[i+1])] += 1
    return pair_counts, symbol_counts


# Compute likelihood score (WordPiece criterion)

In [9]:
def get_best_pair_wordpiece(pair_counts, symbol_counts):
    scores = {}
    for (a, b), count in pair_counts.items():
        scores[(a, b)] = count / (symbol_counts[a] * symbol_counts[b])
    best = max(scores, key=scores.get)
    return best, scores[best]



# merge the best pair

In [5]:
def merge_pair(words, best_pair):
    a, b = best_pair
    new_token = a + b
    new_words = []
    for word in words:
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and word[i] == a and word[i+1] == b:
                new_word.append(new_token)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_words.append(new_word)
    return new_words


In [6]:
vocab = set(ch for word in tokenized for ch in word)

words = tokenized
num_merges = 20

for i in range(num_merges):
    pair_counts, symbol_counts = get_pair_counts(words)
    best_pair, score = get_best_pair_wordpiece(pair_counts, symbol_counts)
    words = merge_pair(words, best_pair)
    vocab.add(''.join(best_pair))
    print(f"Iteration {i+1}: merged {best_pair} (score={score:.6f})")

print("\nFinal vocabulary:")
print(sorted(vocab))


Iteration 1: merged ('q', 'u') (score=0.333333)
Iteration 2: merged ('l', 'y') (score=0.250000)
Iteration 3: merged ('a', 'r') (score=0.142857)
Iteration 4: merged ('c', 'a') (score=0.150000)
Iteration 5: merged ('qu', 'i') (score=0.142857)
Iteration 6: merged ('b', 'o') (score=0.125000)
Iteration 7: merged ('bo', 'y') (score=0.333333)
Iteration 8: merged ('d', 'o') (score=0.160000)
Iteration 9: merged ('n', 'd') (score=0.200000)
Iteration 10: merged ('a', 'nd') (score=0.500000)
Iteration 11: merged ('o', 'n') (score=0.250000)
Iteration 12: merged ('i', 'n') (score=0.166667)
Iteration 13: merged ('a', 's') (score=0.111111)
Iteration 14: merged ('as', 'in') (score=0.333333)
Iteration 15: merged ('u', 'g') (score=0.100000)
Iteration 16: merged ('in', 'g') (score=0.125000)
Iteration 17: merged ('do', 'g') (score=0.166667)
Iteration 18: merged ('asin', 'g') (score=0.500000)
Iteration 19: merged ('ug', 'g') (score=0.500000)
Iteration 20: merged ('ugg', 'ing') (score=0.500000)

Final vocabul

In [10]:
# 🔹 WordPiece Tokenization Step (without ##)
def wordpiece_tokenize(sentence, vocab):
    tokens = []
    for word in sentence.lower().split():
        if word in vocab:
            tokens.append(word)
        else:
            # Try to break the word into subwords from vocab
            sub_tokens = []
            start = 0
            while start < len(word):
                end = len(word)
                found = None
                while start < end:
                    substr = word[start:end]
                    if substr in vocab:
                        found = substr
                        break
                    end -= 1
                if found is None:
                    sub_tokens.append("[UNK]")
                    break
                sub_tokens.append(found)
                start = end
            tokens.extend(sub_tokens)
    return tokens

# Example test sentence
test_sentence = "The cat is chasing the dog quietly."

# Assume vocab is your final vocabulary after 20 merges
tokenized_output = wordpiece_tokenize(test_sentence, vocab)
print("Tokenized Sentence:", tokenized_output)


Tokenized Sentence: ['t', 'h', 'e', 'ca', 't', 'i', 's', 'c', 'h', 'asing', 't', 'h', 'e', 'dog', 'qui', 'e', 't', 'ly', '[UNK]']


# Q : 3

# Preprocessing function (lowercase + tokenize)

In [7]:
import re

def preprocess(sentence):
    sentence = sentence.lower()
    tokens = re.findall(r'\w+|[^\w\s]', sentence)
    return tokens

train_sentences = [
    "Check out https://example.com for more info!",
    "Order 3 items, get 1 free! Limited offer!!!",
    "Your package #12345 will arrive tomorrow.",
    "Win $1000 now, visit http://winbig.com!!!",
    "Meeting at 3pm, don't forget to bring the files.",
    "Exclusive deal for you: buy 2, get 1 free!!!",
    "Download the report from https://reports.com.",
    "The meeting is starting in 10 minutes.",
    "Reminder: submit your timesheet by 5pm today."
]

preprocessed_sentences = [preprocess(s) for s in train_sentences]
preprocessed_sentences


[['check',
  'out',
  'https',
  ':',
  '/',
  '/',
  'example',
  '.',
  'com',
  'for',
  'more',
  'info',
  '!'],
 ['order',
  '3',
  'items',
  ',',
  'get',
  '1',
  'free',
  '!',
  'limited',
  'offer',
  '!',
  '!',
  '!'],
 ['your', 'package', '#', '12345', 'will', 'arrive', 'tomorrow', '.'],
 ['win',
  '$',
  '1000',
  'now',
  ',',
  'visit',
  'http',
  ':',
  '/',
  '/',
  'winbig',
  '.',
  'com',
  '!',
  '!',
  '!'],
 ['meeting',
  'at',
  '3pm',
  ',',
  'don',
  "'",
  't',
  'forget',
  'to',
  'bring',
  'the',
  'files',
  '.'],
 ['exclusive',
  'deal',
  'for',
  'you',
  ':',
  'buy',
  '2',
  ',',
  'get',
  '1',
  'free',
  '!',
  '!',
  '!'],
 ['download',
  'the',
  'report',
  'from',
  'https',
  ':',
  '/',
  '/',
  'reports',
  '.',
  'com',
  '.'],
 ['the', 'meeting', 'is', 'starting', 'in', '10', 'minutes', '.'],
 ['reminder', ':', 'submit', 'your', 'timesheet', 'by', '5pm', 'today', '.']]

# extract bigrams and counts 

In [8]:
from collections import Counter

def get_bigrams(tokens):
    return [ (tokens[i], tokens[i+1]) for i in range(len(tokens)-1) ]

train_bigrams = [get_bigrams(s) for s in preprocessed_sentences]


# compute probabilities with add K smoothing (k=0.3)

In [9]:
K = 0.3

# Combine sentences by class
class_sentences = {
    'Inform': [],
    'Promo': [],
    'Reminder': []
}

labels = ['Inform', 'Promo', 'Inform', 'Promo', 'Reminder', 'Promo', 'Inform', 'Reminder', 'Reminder']

for sent, label in zip(train_bigrams, labels):
    class_sentences[label].extend(sent)

# Count bigrams per class
class_bigram_counts = {c: Counter(class_sentences[c]) for c in class_sentences}

# Vocabulary: all unique bigrams
all_bigrams = set()
for counts in class_bigram_counts.values():
    all_bigrams.update(counts.keys())

V = len(all_bigrams)  # vocabulary size

# Compute P(bigram|class) with add-K smoothing
class_bigram_probs = {}
for c in class_sentences:
    total_count = sum(class_bigram_counts[c].values())
    class_bigram_probs[c] = {}
    for bigram in all_bigrams:
        count = class_bigram_counts[c][bigram]
        prob = (count + K) / (total_count + K * V)
        class_bigram_probs[c][bigram] = prob


# Add URL/number/punctuation features

In [10]:
def extract_features(tokens):
    url = 1 if any('http' in t for t in tokens) else 0
    number = 1 if any(t.isdigit() for t in tokens) else 0
    exclam = tokens.count('!')
    return {'url': url, 'number': number, 'exclam': exclam}


In [11]:
test_sentence = "You will get an exclusive offer in the meeting!"
test_tokens = preprocess(test_sentence)
test_bigrams = get_bigrams(test_tokens)
test_feats = extract_features(test_tokens)


# naive bayes classification

In [14]:
import math

# Prior probabilities
class_counts = Counter(labels)
total = len(labels)
class_priors = {c: class_counts[c]/total for c in class_counts}

# Initialize log-probs
class_scores = {}
for c in class_counts:
    log_prob = math.log(class_priors[c])
    # Add bigram log-probabilities
    for bigram in test_bigrams:
        if bigram in class_bigram_probs[c]:
            log_prob += math.log(class_bigram_probs[c][bigram])
        else:
            # unseen bigram smoothing
            log_prob += math.log(K / (sum(class_bigram_counts[c].values()) + K * V))
    # Incorporate URL/number/exclamation (simplified: +1 if matches class tendency)
    if test_feats['url'] and c == 'Promo':
        log_prob += math.log(0.9)  # heuristic boost
    if test_feats['number'] and c == 'Promo':
        log_prob += math.log(0.8)
    if test_feats['exclam'] > 0 and c == 'Promo':
        log_prob += math.log(0.9)
    class_scores[c] = log_prob

# Predict class
predicted_class = max(class_scores, key=class_scores.get)
predicted_class


'Reminder'